In [ ]:
from datetime import datetime

from matplotlib import pyplot as plt
from matplotlib.axes import Axes
from matplotlib.cm import ScalarMappable
from matplotlib.collections import LineCollection, PolyCollection, QuadMesh
from matplotlib.colorbar import Colorbar
from matplotlib.colors import Normalize
from matplotlib.lines import Line2D
from numpy import arange, ndarray

from utils.py_eddy_tracker import data
from utils.py_eddy_tracker.dataset.grid import RegularGridDataset
from utils.py_eddy_tracker.eddy_feature import Contours
from utils.py_eddy_tracker.observations.observation import EddiesObservations
from visualizer import get_axes, update_axes

### Display sample dataset

In [ ]:
XLIM: tuple[int, int] = (279, 304)
YLIM: tuple[int, int] = (29, 44)

margin: int = 30
g: RegularGridDataset = RegularGridDataset(
    data.get_demo_path("nrt_global_allsat_phy_l4_20190223_20190226.nc"),
    'longitude',
    'latitude',
    indexs={
        'longitude': slice(1116 - margin, 1216 + margin),
        'latitude': slice(476 - margin, 536 + margin),
    },
)

ax: Axes = get_axes("ATD (m)", XLIM, YLIM)
m: QuadMesh = g.display(ax, "adt", vmin=-1, vmax=1, cmap='RdBu_r')

# Draw line on the gulf stream front
great_current: Contours = Contours(g.x_c, g.y_c, g.grid('adt'), levels=(0.35,), keep_unclose=True)
great_current.display(ax, color='k')
update_axes(ax, m)

plt.show()

# 1. look at nrt_global_allsat_phy_l4_20190223_20190226.nc
# 2. get daily / more frequent SWOT data
# 3. Plot that data visually first -> apply py eddy tracker
# 4. Look at PET visualization tools for eddy movement

### Get geostrophic u,v

- calculates the gradient: $u = -(g/f) \times (\partial h / \partial y)$
    - $\frac{\partial h}{\partial x}$ is the slope in the north-south direction, and $\frac{\partial h}{\partial y}$ is the slope in the east-west direction. $u$ and $v$ are the resulting water velocities in the east-west direction and the north-south direction, correspondingly.
- works worse when very close to equation (+/- 2 degrees)

In [ ]:
g.add_uv("adt") # using adt, sla, ssh, or ssha; not necessary if u/v are already pre-computed in the dataset

### Preprocessing

- Large-scale is > 500 km
- Mesoscale is 50-500 km
- Submesoscale is < 50 km

A high-pass filter removes the background signals (i.e., Gulf Stream creates height gradient by itself) and leaves behind only the eddies. Choose bessel because it is a linear transformation applied uniformly, so that shapes of eddies remain the same.

For example, this reveals red anticyclones more clearly in the Northern region and blue cyclones previously-not seen in the Southern region.

In [ ]:
# Apply high-pass filter
g.bessel_high_filter('adt', 700)
ax: Axes = get_axes('ADT (m) filtered (700 km)', XLIM, YLIM)
m: QuadMesh = g.display(ax, 'adt', vmin=-0.4, vmax=0.4, cmap="RdBu_r")
great_current.display(ax, color='k')
update_axes(ax, m)

### Identification

Step indicates contours are drawn every `step` meters (look for closed loops).
- Smaller step -> more/smaller eddies but slower

`shape_error` is the percent deviation acceptance apart from a perfect circle.

In [ ]:
date: datetime = datetime(2016, 5, 15)
a: EddiesObservations
c: EddiesObservations
a, c = g.eddy_identification('adt', 'u', 'v', date, step=0.003, shape_error=55)

In [ ]:
# Display all contours (eddy or not) only; no colored background
ax: Axes = get_axes('ADT closed contours (one 1 per 5 levels)', XLIM, YLIM)
g.contours.display(ax, step=5, lw=1) # type: ignore
great_current.display(ax, color='k')
sm: ScalarMappable = ScalarMappable(norm=Normalize(vmin=-0.4, vmax=0.4), cmap='RdYlBu_r')
cbar: Colorbar = plt.colorbar(sm, cax=ax.figure.add_axes((0.94, 0.05, 0.01, 0.9)))
cbar.set_label('ADT (m)')

In [ ]:
# Display only contours within eddies
ax: Axes = get_axes('ADT closed contours (only those in eddy)', XLIM, YLIM)
g.contours.display(ax, lw=0.25, only_used=True) # type: ignore
great_current.display(ax, color='k')
update_axes(ax)

### Post-Analysis

##### Eddy Rejection Reasons

- Green: accepted
- Red: rejected due to shape error
- Blue: masked or invalid values inside of contour
    - If any inside pixel is invalid / NaN, then this issue is raised
    - Preprocess + interpolate if needed
- Black: too few or too many pixels (in terms of a reasonable size that an eddy could be)
- Yellow: failed amplitude criterion (difference between center height and edge)

In [ ]:
ax: Axes = get_axes('Eddy Rejection Reasons', XLIM, YLIM)
g.contours.display(ax, only_unused=True, display_criterion=True, lw=0.25) # type: ignore
great_current.display(ax, color='k')
update_axes(ax)

##### Shape Error of All Contours

In [ ]:
# field must be 'shape_error', 'x', 'y' or 'radius'.
# If defined display_criterion is not used.
# bins argument must be defined

ax: Axes = get_axes('Contour Shape Error', XLIM, YLIM)
m: LineCollection = g.contours.display(ax, lw=0.5, field='shape_error', bins=arange(0, 105, 5), cmap='PRGn_r') # type: ignore
update_axes(ax, m)

##### Contours Containing >1 Eddy

In [ ]:
ax: Axes = get_axes('ADT rejected contours containing eddies', XLIM, YLIM)
g.contours.label_contour_unused_which_contain_eddies(a) #type:ignore
g.contours.label_contour_unused_which_contain_eddies(c) #type:ignore

# Black contour for containing multiple eddies
# Contour itself wasn't selected as an eddy boundary but contains them
# All unused contours at any height that contains eddies inside
g.contours.display( #type:ignore
    ax,
    only_contain_eddies=True,
    color='k',
    lw=1,
    label='could be contour of interaction'
) 

a.display(ax, color='r', linewidth=0.75, label='anticyclone', ref=-10)
c.display(ax, color='b', linewidth=0.75, label='cyclone', ref=-10)
ax.legend()
update_axes(ax)

##### Displaying Outputs

Dashed line style is an effective contour, which is the outer edge of the eddy (where it ends).

Solid line is the max mean speed, which is where rotation is the fastest (core).

Essentially, at the center of the eddy the rotation is almost 0. And then it increases moving outward to a specifc radius (approaches max). Then moving outward further it decreases.

In [ ]:
ax: Axes = get_axes('Eddies detected', XLIM, YLIM)
a.display(
    ax,
    color='r',
    linewidth=0.75,
    label='anticyclonic ({nb_obs} eddies)',
    ref=-10
)
c.display(
    ax,
    color='b',
    linewidth=0.75,
    label='cyclonic ({nb_obs} eddies)',
    ref=-10
)
ax.legend()
great_current.display(ax, color='k')
update_axes(ax)

Display the effective radius of eddies

In [ ]:
ax: Axes = get_axes('Effective radius (km)', XLIM, YLIM)
a.filled(
    ax,
    'radius_e',
    vmin=10,
    vmax=150,
    cmap='magma_r',
    factor=0.001, # radius is stored in meters initially; displayed as radius_e * factor -> km
    lut=14 # lookup table; # of discrete colors in the cmap
)
c.filled(
    ax,
    'radius_e',
    vmin=10,
    vmax=150,
    cmap='magma_r',
    factor=0.001,
    lut=14
)
great_current.display(ax, color='k')
update_axes(ax)

In [ ]:
plt.close('all')